# Part 1: GPU-Accelerated XGBoost Training

**High-Performance Model Training on GPU**

In this notebook, you'll:
- Load the feature-engineered data from Module 3
- Train XGBoost models using GPU acceleration
- Perform cross-validation and hyperparameter tuning
- Evaluate model performance and feature importance
- Generate predictions for submission

**Libraries Used:**
- `XGBoost` with GPU support for fast training
- `scikit-learn` for model evaluation
- `matplotlib` and `seaborn` for visualization
- `SHAP` for model explainability

## 1. Environment Setup

In [36]:
%load_ext cudf.pandas

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [57]:
import time
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd


# XGBoost and sklearn
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import shap

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Data paths
DATA_DIR = Path("./data")
FEATURE_DIR = DATA_DIR / "features"
FEATURE_DIR.mkdir(exist_ok=True)
MODEL_DIR = DATA_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

print(f"XGBoost version: {xgb.__version__}")
print(f"\nData directory: {DATA_DIR}")
print(f"\nFeature directory: {FEATURE_DIR}")
print(f"Model directory: {MODEL_DIR}")


XGBoost version: 3.1.2

Data directory: data

Feature directory: data/features
Model directory: data/models


In [38]:
pd.set_option("display.max_rows", None)       # show all rows
pd.set_option("display.max_columns", None)    # show all columns
pd.set_option("display.max_colwidth", None)   # don't truncate cell content
pd.set_option("display.width", None)          # no line-wrap
pd.set_option("display.float_format", "{:.4f}".format)  # optional: decimal places

## 2. Load Feature-Engineered Data

In [39]:
# Load engineered features
print("Loading feature-engineered data...")
start = time.perf_counter()

train_df = pd.read_parquet(FEATURE_DIR / "train_features.parquet")
test_df = pd.read_parquet(FEATURE_DIR / "test_features.parquet")

# Load feature list
with open(FEATURE_DIR / "feature_cols.txt", 'r') as f:
    feature_cols = f.read().strip().split('\n')

# Filter to only numeric features that exist
available_features = [col for col in feature_cols if col in train_df.columns]
feature_cols = available_features

print(f"Load time: {time.perf_counter() - start:.2f}s")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Available features: {len(feature_cols)}")
print(f"Fraud rate: {train_df['isFraud'].mean():.2%}")

Loading feature-engineered data...
Load time: 0.42s
Train shape: (590540, 468)
Test shape: (506691, 467)
Available features: 466
Fraud rate: 3.50%


## 3. Data Preparation

In [40]:
# Prepare features and target
X = train_df[feature_cols]
y = train_df['isFraud']

X_test = test_df[feature_cols]
test_ids = test_df['TransactionID']

print(f"Training data shape: {X.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"\nClass distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.value_counts(normalize=True)}")

Training data shape: (590540, 466)
Test data shape: (506691, 466)

Class distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Class balance: isFraud
0   0.9650
1   0.0350
Name: proportion, dtype: float64


## 4. Baseline Model Training

Let's start with a simple baseline model using default XGBoost parameters with GPU acceleration.

In [41]:
# Split data for validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Training fraud rate: {y_train.mean():.2%}")
print(f"Validation fraud rate: {y_val.mean():.2%}")

Training set: (472432, 466)
Validation set: (118108, 466)
Training fraud rate: 3.50%
Validation fraud rate: 3.50%


In [42]:
# Handle imbalanced data
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f"\nClass distribution in training set:")
print(f"Legit transactions: {neg}")
print(f"Fraud transactions: {pos}")
print(f"scale_pos_weight:   {scale:.2f}")


Class distribution in training set:
Legit transactions: 455902
Fraud transactions: 16530
scale_pos_weight:   27.58


In [43]:
%%time

# Baseline XGBoost parameters with GPU
baseline_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda',  # ← Changed from 'cuda' to 'cpu'
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbosity': 1,
    'scale_pos_weight': scale
}

# Training
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

baseline_model = xgb.train(
    baseline_params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=50
)

[0]	train-auc:0.84280	val-auc:0.84109
[50]	train-auc:0.92509	val-auc:0.91555
[100]	train-auc:0.95059	val-auc:0.93496
[150]	train-auc:0.96447	val-auc:0.94534
[200]	train-auc:0.97345	val-auc:0.95179
[250]	train-auc:0.97970	val-auc:0.95649
[300]	train-auc:0.98359	val-auc:0.95915
[350]	train-auc:0.98711	val-auc:0.96191
[400]	train-auc:0.98965	val-auc:0.96407
[450]	train-auc:0.99174	val-auc:0.96546
[499]	train-auc:0.99334	val-auc:0.96682
CPU times: user 4.54 s, sys: 521 ms, total: 5.06 s
Wall time: 4.83 s


In [44]:
# Evaluate baseline model
y_pred_train = baseline_model.predict(dtrain)
y_pred_val = baseline_model.predict(dval)

train_auc = roc_auc_score(y_train, y_pred_train)
val_auc = roc_auc_score(y_val, y_pred_val)

print("\n" + "="*60)
print("📊 BASELINE MODEL PERFORMANCE")
print("="*60)
print(f"Training AUC:   {train_auc:.5f}")
print(f"Validation AUC: {val_auc:.5f}")
print(f"Best iteration: {baseline_model.best_iteration}")
print("="*60)


📊 BASELINE MODEL PERFORMANCE
Training AUC:   0.99334
Validation AUC: 0.96682
Best iteration: 499


## 5. HPO
Perform ml model fine tuning using hyperparameter optimization (HPO) 

In [ ]:
import optuna
from optuna.integration import DaskStorage
from dask.distributed import Client, wait
from dask_cuda import LocalCUDACluster

# This will use all GPUs on the local host by default
cluster = LocalCUDACluster(threads_per_worker=1, ip="", dashboard_address="8081")
client = Client(cluster)

# Query the client for all connected workers
workers = client.has_what().keys()
n_workers = len(workers)
client

Connection method: Cluster object,Cluster type: dask_cuda.LocalCUDACluster
Dashboard: http://10.110.42.107:43987/status,
Dashboard: http://10.110.42.107:43987/status,Workers: 2
Total threads: 2,Total memory: 125.48 GiB
Status: running,Using processes: True
Comm: tcp://10.110.42.107:43467,Workers: 0
Dashboard: http://10.110.42.107:43987/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://10.110.42.107:46009,Total threads: 1
Dashboard: http://10.110.42.107:37033/status,Memory: 62.74 GiB
Nanny: tcp://10.110.42.107:38547,


[I 2026-05-04 15:53:20,545] A new study created in memory with name: dask_optuna_xgboost
[I 2026-05-04 15:54:34,089] A new study created in memory with name: dask_optuna_xgboost
[I 2026-05-04 15:56:49,912] A new study created in memory with name: dask_optuna_xgboost
2026-05-04 16:08:00,906 - distributed.scheduler - WARNING - Removing worker 'tcp://10.110.42.107:35763' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'optimize-c595d878-2443-42b7-b53f-40b8f074e378'} (stimulus_id='handle-worker-cleanup-1777936080.9062917')
2026-05-04 16:08:00,907 - distributed.scheduler - WARNING - Removing worker 'tcp://10.110.42.107:46009' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'optimize-883ec902-215b-4e7a-a47f-a1bc20bcabef'} (stimulus_id='handle-worker-cleanup-1777936080.90723')


In [46]:
# Number of trials for optimization
N_TRIALS = 50

# ensure all arrays are plain numpy before sending to Dask workers
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

# Study name for Optuna
study_name = "dask_optuna_xgboost"

In [47]:
def train_and_eval_xgbc(X_train, X_val, y_train, y_val, 
                        n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                        colsample_bytree=0.8, reg_alpha=0.0, reg_lambda=1.0, min_child_weight=1.0):
    """
    Evaluate XGBoost classifier with given hyperparameters on the provided training and validation data.

    """

    # Baseline XGBoost parameters with GPU
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    baseline_params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'device': 'cuda',
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'seed': 42,
        'verbosity': 1,
        'scale_pos_weight': scale,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda, 
        'min_child_weight': min_child_weight
    }

    baseline_model = xgb.train(
        baseline_params,
        dtrain,
        num_boost_round=n_estimators,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=50,
        verbose_eval=50
    )
    
    # y_prob = baseline_model.predict(dtest)  # Use ROC AUC score for evaluation
    # score = roc_auc_score(y_test, y_prob)

    y_pred = [1 if p >= 0.5 else 0 for p in baseline_model.predict(dval)]
    score = f1_score(y_val, y_pred)
    
    return score

In [48]:
print("Default XGBoost F1 Score:", train_and_eval_xgbc(X_train, X_val, y_train, y_val))

[0]	train-auc:0.84280	val-auc:0.84109
[50]	train-auc:0.90450	val-auc:0.89784
[100]	train-auc:0.92440	val-auc:0.91486
[150]	train-auc:0.93981	val-auc:0.92661
[200]	train-auc:0.94970	val-auc:0.93435
[250]	train-auc:0.95764	val-auc:0.93960
[299]	train-auc:0.96436	val-auc:0.94459
Default XGBoost F1 Score: 0.40252020922491677


In [49]:
def objective(trial, X_train, X_val, y_train, y_val):

    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 3, 8)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)

    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0)

    reg_alpha = trial.suggest_float("reg_alpha", 1e-6, 1.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True)
    min_child_weight = trial.suggest_float("min_child_weight", 1, 10, log=True)

    score = train_and_eval_xgbc(
        X_train, 
        X_val, 
        y_train, 
        y_val,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        min_child_weight=min_child_weight,
    )
    
    return score

In [52]:
X_train_np = np.asarray(X_train, dtype=np.float32)
X_val_np = np.asarray(X_val, dtype=np.float32)
y_train_np = np.asarray(y_train).ravel()
y_val_np = np.asarray(y_val).ravel()


# Set up Dask client for distributed computation
storage = DaskStorage()

# Create Optuna study
study = optuna.create_study(
    sampler=optuna.samplers.TPESampler(seed=142),
    study_name=study_name,
    direction="maximize",  # Maximize ROC AUC score
    storage=storage,
)

# Optimize in parallel on your Dask cluster
n_workers = 2  # Adjust based on your available resources
futures = [
    client.submit(
        study.optimize,
        lambda trial: objective(trial, X_train_np, X_val_np, y_train_np, y_val_np),
        n_trials=N_TRIALS // n_workers,
        pure=False,
    )
    for _ in range(n_workers)
]
wait(futures)

# Output results
print(f"Best params: {study.best_params}")
print("Best F1 score: ", study.best_value)  # Convert back to positive score
print("Number of finished trials: ", len(study.trials))


[0]	train-auc:0.83452	val-auc:0.83321
[50]	train-auc:0.92020	val-auc:0.91169
[0]	train-auc:0.83452	val-auc:0.83321
[100]	train-auc:0.94337	val-auc:0.92948
[150]	train-auc:0.95802	val-auc:0.94051
[50]	train-auc:0.92020	val-auc:0.91169
[200]	train-auc:0.96776	val-auc:0.94774
[250]	train-auc:0.97460	val-auc:0.95309
[300]	train-auc:0.97911	val-auc:0.95660
[100]	train-auc:0.94337	val-auc:0.92948
[350]	train-auc:0.98267	val-auc:0.95905
[400]	train-auc:0.98576	val-auc:0.96153
[150]	train-auc:0.95802	val-auc:0.94051
[450]	train-auc:0.98794	val-auc:0.96343
[460]	train-auc:0.98850	val-auc:0.96379


[I 2026-05-04 15:56:55,945] Trial 0 finished with value: 0.5198931116389549 and parameters: {'n_estimators': 461, 'max_depth': 6, 'learning_rate': 0.09310413107735414, 'subsample': 0.9162357063860624, 'colsample_bytree': 0.4399189310911876, 'reg_alpha': 5.801302946369961e-06, 'reg_lambda': 2.062136084439984, 'min_child_weight': 2.705755514095741}. Best is trial 0 with value: 0.5198931116389549.


[200]	train-auc:0.96776	val-auc:0.94774
[250]	train-auc:0.97460	val-auc:0.95309
[300]	train-auc:0.97911	val-auc:0.95660
[350]	train-auc:0.98267	val-auc:0.95905
[400]	train-auc:0.98576	val-auc:0.96153
[0]	train-auc:0.84704	val-auc:0.84131
[50]	train-auc:0.94995	val-auc:0.93328
[450]	train-auc:0.98794	val-auc:0.96343
[460]	train-auc:0.98850	val-auc:0.96379
[100]	train-auc:0.97372	val-auc:0.94976


[I 2026-05-04 15:56:59,255] Trial 1 finished with value: 0.5198931116389549 and parameters: {'n_estimators': 461, 'max_depth': 6, 'learning_rate': 0.09310413107735414, 'subsample': 0.9162357063860624, 'colsample_bytree': 0.4399189310911876, 'reg_alpha': 5.801302946369961e-06, 'reg_lambda': 2.062136084439984, 'min_child_weight': 2.705755514095741}. Best is trial 0 with value: 0.5198931116389549.


[150]	train-auc:0.98327	val-auc:0.95596
[200]	train-auc:0.98894	val-auc:0.96022
[250]	train-auc:0.99281	val-auc:0.96334
[253]	train-auc:0.99297	val-auc:0.96338


...

[I 2026-05-04 16:00:09,218] Trial 47 finished with value: 0.7384683783483383 and parameters: {'n_estimators': 363, 'max_depth': 8, 'learning_rate': 0.2603775004195342, 'subsample': 0.5619498776130805, 'colsample_bytree': 0.34947758073866736, 'reg_alpha': 8.990781217337481e-05, 'reg_lambda': 0.09603987878099009, 'min_child_weight': 1.378508530528301}. Best is trial 32 with value: 0.7952472395583293.


[0]	train-auc:0.86229	val-auc:0.84585
[50]	train-auc:0.98152	val-auc:0.95199
[100]	train-auc:0.99408	val-auc:0.96225
[150]	train-auc:0.99801	val-auc:0.96679
[200]	train-auc:0.99926	val-auc:0.96856
[250]	train-auc:0.99974	val-auc:0.96969
[300]	train-auc:0.99991	val-auc:0.97052
[329]	train-auc:0.99995	val-auc:0.97076


[I 2026-05-04 16:00:17,916] Trial 48 finished with value: 0.7727113702623907 and parameters: {'n_estimators': 330, 'max_depth': 8, 'learning_rate': 0.25898044434471906, 'subsample': 0.7738104196374515, 'colsample_bytree': 0.31851002920985017, 'reg_alpha': 0.00022410953041521886, 'reg_lambda': 0.026389764988663408, 'min_child_weight': 1.9630762123183711}. Best is trial 32 with value: 0.7952472395583293.


[0]	train-auc:0.86155	val-auc:0.84808
[50]	train-auc:0.97968	val-auc:0.95140
[100]	train-auc:0.99411	val-auc:0.96253
[150]	train-auc:0.99785	val-auc:0.96589
[200]	train-auc:0.99912	val-auc:0.96727
[250]	train-auc:0.99972	val-auc:0.96834
[300]	train-auc:0.99990	val-auc:0.96918
[319]	train-auc:0.99994	val-auc:0.96956
Best params: {'n_estimators': 493, 'max_depth': 8, 'learning_rate': 0.2021038547507557, 'subsample': 0.5356138409033132, 'colsample_bytree': 0.5709430850336852, 'reg_alpha': 9.284194506306841e-06, 'reg_lambda': 0.06863350026428341, 'min_child_weight': 2.60246496595116}
Best F1 score:  0.7952472395583293
Number of finished trials:  50


[I 2026-05-04 16:00:26,486] Trial 49 finished with value: 0.7742085121187522 and parameters: {'n_estimators': 320, 'max_depth': 8, 'learning_rate': 0.25419240271035176, 'subsample': 0.8014832865269091, 'colsample_bytree': 0.3006568988295641, 'reg_alpha': 0.0001974047381285958, 'reg_lambda': 0.01022864679479465, 'min_child_weight': 1.4182284974690738}. Best is trial 32 with value: 0.7952472395583293.


## 6. Final Model Training

Train the final model using the best parameters on the full training set.

In [ ]:
# Best parameters (update based on tuning results)
from sklearn.metrics import precision_score


final_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda',
    'max_depth': 8,
    'learning_rate': 0.2021038547507557,
    'subsample': 0.5356138409033132,
    'colsample_bytree': 0.5709430850336852,
    'seed': 42,
    'verbosity': 1,
    'scale_pos_weight': scale,
    'reg_alpha': 9.284194506306841e-06,
    'reg_lambda': 0.06863350026428341, 
    'min_child_weight': 2.60246496595116
    }

print("Training final model with best parameters...")
print("Parameters:", final_params)
print()

# Use training and test data
dtrain = xgb.DMatrix(X_train_np, label=y_train_np)
dval = xgb.DMatrix(X_val_np, label=y_val_np)

final_model = xgb.train(
    final_params,
    dtrain,
    num_boost_round=320,
    evals=[(dtrain, 'train'), (dval, 'val')],
    verbose_eval=100
)

y_pred = [1 if p >= 0.5 else 0 for p in final_model.predict(dval)]
f1 = f1_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
roc_auc = roc_auc_score(y_val, final_model.predict(dval))
print(f"Final Model F1 Score: {f1}")
print(f"Final Model Recall: {recall}")
print(f"Final Model Precision: {precision}")
print(f"Final Model ROC AUC: {roc_auc}")


Training final model with best parameters...
Parameters: {'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda', 'max_depth': 8, 'learning_rate': 0.25419240271035176, 'subsample': 0.8014832865269091, 'colsample_bytree': 0.3006568988295641, 'seed': 42, 'verbosity': 1, 'scale_pos_weight': np.float64(27.580278281911674), 'reg_alpha': 0.0001974047381285958, 'reg_lambda': 0.01022864679479465, 'min_child_weight': 1.4182284974690738}

[0]	train-auc:0.86155	val-auc:0.84808
[100]	train-auc:0.99411	val-auc:0.96253
[200]	train-auc:0.99912	val-auc:0.96727
[300]	train-auc:0.99990	val-auc:0.96918
[319]	train-auc:0.99994	val-auc:0.96956
Final Model F1 Score: 0.7742085121187522
Final Model Recall: 0.8076457778853133
Final Model Precision: 0.7434298440979955
Final Model ROC AUC: 0.9695630475333036


In [59]:
# Save the final model
model_path = MODEL_DIR / "xgboost_final_model.json"
final_model.save_model(str(model_path))
print(f"Model saved to: {model_path}")

Model saved to: data/models/xgboost_final_model.json


## 7. Generate Predictions

In [62]:
# Generate predictions on test set
print("Generating predictions on test set...")
dtest = xgb.DMatrix(X_test)
test_predictions = final_model.predict(dtest)

print(f"\nPrediction statistics:")
print(f"  Min:    {test_predictions.min():.6f}")
print(f"  Max:    {test_predictions.max():.6f}")
print(f"  Mean:   {test_predictions.mean():.6f}")
print(f"  Median: {np.median(test_predictions):.6f}")
print(f"\nPredicted fraud rate: {(test_predictions > 0.5).mean():.2%}")

Generating predictions on test set...

Prediction statistics:
  Min:    0.000000
  Max:    1.000000
  Mean:   0.031236
  Median: 0.000766

Predicted fraud rate: 2.17%
